# 01 — Data Extraction & Cleaning

Pipeline stage 1 of 5.

Goals:
1. Load CTI documents (PDF / DOCX / TXT / HTML / MD) from `uploads/`
2. Load the labelled training corpus (`dataset/synthetic_cti_reports.csv`)
3. Detect key columns (text, title, label, date)
4. Clean, dedupe, and build a unified `model_text` field

Saves:
- `data/processed/cleaned_reports.pkl`
- `data/processed/uploaded_docs.pkl`


In [13]:

# ---------------------------------------------------------------------------
# Bootstrap — paths, imports, global config
# ---------------------------------------------------------------------------
import os
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(r"D:\Projects\done\thesis\ctc_ectd")
SCRIPTS_DIR  = PROJECT_ROOT / "scripts"
UPLOAD_DIR   = PROJECT_ROOT / "uploads"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

for d in (SCRIPTS_DIR, UPLOAD_DIR, ARTIFACT_DIR, PROCESSED_DIR):
    d.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")

print("PROJECT_ROOT :", PROJECT_ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("ARTIFACT_DIR :", ARTIFACT_DIR)
print("UPLOAD_DIR   :", UPLOAD_DIR)


PROJECT_ROOT : D:\Projects\done\thesis\ctc_ectd
PROCESSED_DIR: D:\Projects\done\thesis\ctc_ectd\data\processed
ARTIFACT_DIR : D:\Projects\done\thesis\ctc_ectd\artifacts
UPLOAD_DIR   : D:\Projects\done\thesis\ctc_ectd\uploads


## 1. Document extraction (uploads/)

In [14]:

from document_extractor import (
    extract_text,
    clean_extracted_text,
    extract_and_clean,
    EXTRACTORS,
)

print("Supported extensions:", ", ".join(sorted(EXTRACTORS)))


def load_documents(folder: Path) -> pd.DataFrame:
    """Walk `folder` and extract text from every supported document."""
    rows = []
    for path in sorted(folder.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix.lower() not in EXTRACTORS:
            continue
        try:
            raw = extract_text(path)
            cleaned = clean_extracted_text(raw)
            rows.append({
                "file_name": path.name,
                "file_path": str(path),
                "extension": path.suffix.lower(),
                "char_count": len(cleaned),
                "word_count": len(cleaned.split()),
                "raw_text": raw,
                "text": cleaned,
            })
        except Exception as exc:
            print(f"[WARN] {path.name}: {exc}")
            rows.append({
                "file_name": path.name,
                "file_path": str(path),
                "extension": path.suffix.lower(),
                "char_count": 0,
                "word_count": 0,
                "raw_text": "",
                "text": "",
            })
    return pd.DataFrame(rows)


Supported extensions: .csv, .doc, .docx, .htm, .html, .log, .md, .pdf, .txt


In [15]:

uploaded_df = load_documents(UPLOAD_DIR)

if uploaded_df.empty:
    print("No documents found in", UPLOAD_DIR)
    print("Writing empty uploaded_docs.pkl so downstream notebooks do not break.")
    uploaded_df = pd.DataFrame(columns=[
        "file_name", "file_path", "extension",
        "char_count", "word_count", "raw_text", "text",
    ])
else:
    print(f"Extracted text from {len(uploaded_df)} document(s).")
    display(uploaded_df[["file_name", "extension", "char_count", "word_count"]]
            .sort_values("char_count", ascending=False))

uploaded_df.to_pickle(PROCESSED_DIR / "uploaded_docs.pkl")
print("Saved:", PROCESSED_DIR / "uploaded_docs.pkl")


Extracted text from 10 document(s).


,file_name,extension,char_count,word_count
0,2024-01_phishing_report_01.pdf,.pdf,1058,140
2,2024-05_credential_theft_report_03.pdf,.pdf,1011,137
1,2024-03_ransomware_report_02.pdf,.pdf,989,135
8,2025-05_vulnerability_report_09.pdf,.pdf,984,133
9,2025-07_credential_theft_report_10.pdf,.pdf,984,133
5,2024-11_data_breach_report_06.pdf,.pdf,957,133
6,2025-01_insider_threat_report_07.pdf,.pdf,916,132
7,2025-03_botnet_report_08.pdf,.pdf,901,128
3,2024-07_ddos_report_04.pdf,.pdf,900,129
4,2024-09_web_attack_report_05.pdf,.pdf,898,127


Saved: D:\Projects\done\thesis\ctc_ectd\data\processed\uploaded_docs.pkl


## 2. Load labelled training corpus

In [16]:

DATA_PATH = PROJECT_ROOT / "dataset" / "synthetic_cti_reports.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Place synthetic_cti_reports.csv inside the dataset/ folder."
    )

df = pd.read_csv(DATA_PATH)
print("Training corpus shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())


Training corpus shape: (10000, 17)


,id,title,text,category,date,source,url,severity,threat_actor,malware,attack_vector,target_sector,country,cve_id,mitre_technique,mitre_technique_name,indicators
0,3568,DDoS activity targeting Retail organizations,A recent threat campaign involved ddos activit...,DDoS,2021-01-01,CTI-Source-6,https://cti-reports.example.org/report/3568,High,BlackOrchid,SilverWorm,Drive-by Download,Retail,Japan,CVE-2022-39414,T1059,Command and Scripting Interpreter,IP=111.96.63.206; DOMAIN=wall-west.net; SHA256...
1,5617,Web Attack activity targeting Financial Servic...,A medium web attack campaign targeted organiza...,Web Attack,2021-01-01,CTI-Source-9,https://cti-reports.example.org/report/5617,Medium,DarkHydra,DarkLoader,Brute Force,Financial Services,United States,CVE-2024-29220,T1053,Scheduled Task/Job,IP=150.80.211.45; DOMAIN=austin-moreno.net; SH...
2,2016,Credential Theft activity targeting Retail org...,A high credential theft campaign targeted orga...,Credential Theft,2021-01-01,CTI-Source-11,https://cti-reports.example.org/report/2016,High,StormByte,SilverWorm,Exposed Service,Retail,Australia,CVE-2026-54394,T1566.001,Spearphishing Attachment,IP=143.178.240.66; DOMAIN=briggs.org; SHA256=f...
3,4000,Credential Theft activity targeting Government...,A recent threat campaign involved credential t...,Credential Theft,2021-01-01,CTI-Source-2,https://cti-reports.example.org/report/4000,Critical,NightWolf,RedLineX,Exposed Service,Government,United States,CVE-2026-60798,T1059,Command and Scripting Interpreter,IP=46.62.120.64; DOMAIN=williams.com; SHA256=3...
4,1066,Phishing activity targeting Banking organizations,A high phishing campaign targeted organization...,Phishing,2021-01-01,CTI-Source-1,https://cti-reports.example.org/report/1066,High,APT-Alpha,ShadowRAT,Brute Force,Banking,India,CVE-2023-68334,T1083,File and Directory Discovery,IP=33.169.134.207; DOMAIN=holmes-matthews.org;...



Columns: ['id', 'title', 'text', 'category', 'date', 'source', 'url', 'severity', 'threat_actor', 'malware', 'attack_vector', 'target_sector', 'country', 'cve_id', 'mitre_technique', 'mitre_technique_name', 'indicators']


## 3. Detect key columns

In [17]:

def find_column(columns, candidates):
    lookup = {str(col).strip().lower(): col for col in columns}
    for cand in candidates:
        if cand in lookup:
            return lookup[cand]
    return None


TEXT_COL  = find_column(df.columns,
    ["text", "report_text", "description", "content",
     "narrative", "report", "body"])
TITLE_COL = find_column(df.columns, ["title", "report_title", "name"])
LABEL_COL = find_column(df.columns,
    ["attack_type", "label", "category", "threat_type", "class", "target"])
DATE_COL  = find_column(df.columns,
    ["date", "published_date", "report_date", "created_at", "timestamp"])

print("TEXT_COL :", TEXT_COL)
print("TITLE_COL:", TITLE_COL)
print("LABEL_COL:", LABEL_COL)
print("DATE_COL :", DATE_COL)

if TEXT_COL is None:
    raise ValueError("Text column not detected.")
if LABEL_COL is None:
    raise ValueError("Label column not detected.")


TEXT_COL : text
TITLE_COL: title
LABEL_COL: category
DATE_COL : date


## 4. Clean & dedupe

In [18]:

df = df.copy()
df[TEXT_COL]  = df[TEXT_COL].fillna("").astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].fillna("").astype(str).str.strip()

df = df[(df[TEXT_COL] != "") & (df[LABEL_COL] != "")].copy()
df = df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

if TITLE_COL is not None:
    df[TITLE_COL] = df[TITLE_COL].fillna("").astype(str).str.strip()
    df["model_text"] = (df[TITLE_COL] + " " + df[TEXT_COL]).str.strip()
else:
    df["model_text"] = df[TEXT_COL]

print("Cleaned shape:", df.shape)
display(df[LABEL_COL].value_counts().to_frame("report_count"))


Cleaned shape: (10000, 18)


,report_count
Ransomware,1041
Web Attack,1040
DDoS,1027
Vulnerability,1020
Credential Theft,993
Data Breach,990
Botnet,987
Malware,986
Phishing,983
Insider Threat,933


## 5. Save for downstream notebooks

In [19]:

payload = {
    "df": df,
    "TEXT_COL": TEXT_COL,
    "TITLE_COL": TITLE_COL,
    "LABEL_COL": LABEL_COL,
    "DATE_COL": DATE_COL,
}
pd.to_pickle(payload, PROCESSED_DIR / "cleaned_reports.pkl")
print("Saved:", PROCESSED_DIR / "cleaned_reports.pkl")


Saved: D:\Projects\done\thesis\ctc_ectd\data\processed\cleaned_reports.pkl
